# 02b — Train the reverse en→hi Transformer (for back-translation)

Mirror of `02_train`, **direction flipped**: source = English, target = Hindi. This trains the **reverse** model that back-translation uses to turn monolingual English into synthetic Hindi — the prerequisite for `03_backtranslation`.

**Same as `02_train` otherwise:** same base Transformer, same joint tokenizer (one shared vocab works both directions — no retokenization), same training loop / optimizer / schedule / AMP / EMA / early-stopping. Only two things change: the train/dev **source and target files are swapped**, and checkpoints + TB logs go to **`checkpoints/backtranslate`** on Drive (the forward run lives in `checkpoints/forwardtranslate`).

**Before running:** GPU runtime (Runtime -> Change runtime type -> GPU), and `01_data` must have populated `DATA_ROOT` on Drive (`train.labse.*`, `dev.clean.*`, `spm_unigram_16000.model`) — the *same* files `02_train` uses, just read in the opposite direction.

The loop reports dev **nll** (now predicting Hindi). It **auto-resumes** from the latest `step_*.pt` in the Drive `out_dir`, so a disconnect is fine.

## 1. Repo + install

In [1]:
import os, sys

REPO_URL = "https://github.com/Rishi-Jain-27/hindi-translator.git"
REPO_DIR = "/content/hindi-translator"

!test -d {REPO_DIR} && git -C {REPO_DIR} pull || git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

Cloning into '/content/hindi-translator'...
remote: Enumerating objects: 480, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 480 (delta 20), reused 29 (delta 13), pack-reused 434 (from 1)
Receiving objects: 100% (480/480), 13.77 MiB | 17.30 MiB/s, done.
Resolving deltas: 100% (242/242), done.
cwd: /content/hindi-translator


In [2]:
!pip install -e .

Obtaining file:///content/hindi-translator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 146.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 15.5 MB/s eta 0:00:00
  Building editable for nmt (pyproject.toml) ... done
  Created wheel for nmt: filename=nmt-0.1.0-0.editable-py3-none-any.whl size=1279 sha256=2a63ab439a4dcd4237df962ea56f8e0492ac9a010ee8dfd3c4888c6e5760eaed
  Stored in directory: /tmp/pip-ephem-wheel-cache-lp__7v73/wheels/bd/c2/ef/ae818ff943d77ea8d63ef07aea61a1b82808716362dc169d4c
Successfully built nmt


In [3]:
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

cuda: True NVIDIA A100-SXM4-40GB


## 2. Configs

`ModelConfig` is the same base Transformer (it must match what `04_decode_eval` / `03_backtranslation` will load to back-translate). `TrainConfig` uses the same real-run values as `02_train`; only `out_dir` differs (-> `backtranslate`).

In [6]:
from nmt.config import DataConfig, ModelConfig, TrainConfig
from google.colab import drive

# same corpus + tokenizer 01_data wrote to Drive (read in the opposite direction below)
drive.mount("/content/drive")
DATA_ROOT = "/content/drive/MyDrive/hindi-translator/data"
CKPT_ROOT = "/content/drive/MyDrive/hindi-translator/checkpoints/backtranslate"   # reverse en->hi ckpts (forward lives in forwardtranslate)

data_cfg = DataConfig(raw_dir=f"{DATA_ROOT}/raw", cache_dir=f"{DATA_ROOT}/cache")
model_cfg = ModelConfig()                      # base: d_model 512, n_heads 8, 6+6 layers, d_ff 2048

train_cfg = TrainConfig(
    out_dir=CKPT_ROOT,         # reverse-model checkpoints + TB logs -> Drive/backtranslate (loop auto-resumes)
    # same real-run schedule as 02_train (smoke values: max_steps=300, warmup=100, log=20, eval=100, ckpt=100)
    max_steps=100_000,         # upper bound; patience-based early stopping usually halts sooner
    warmup_steps=4000,         # standard Transformer inverse-sqrt warmup
    eval_every=2000,           # dev-nll eval cadence
    ckpt_every=2000,           # keeps only the last 3 step_*.pt (ckpt_keep_last_n=3) + best.pt
    log_every=50,
)
print("out_dir:", train_cfg.out_dir)
print("eff tokens/step:", train_cfg.max_tokens * train_cfg.grad_accum,
      "| amp:", train_cfg.amp, train_cfg.amp_dtype, "| max_steps:", train_cfg.max_steps)

Mounted at /content/drive
out_dir: /content/drive/MyDrive/hindi-translator/checkpoints/backtranslate
eff tokens/step: 32768 | amp: True auto | max_steps: 100000


## 3. Tokenizer + data loaders — direction flipped

Same files as `02_train`, but the two positional path args to `make_dataloader` are **(source, target)** — so here we pass **English as source** and **Hindi as target** (the opposite of the forward run). The joint tokenizer encodes either language with the same vocab, so nothing else changes.

In [7]:
from nmt.data.tokenizer import Tokenizer
from nmt.data.dataset import make_dataloader

cache = data_cfg.cache_dir
tok = Tokenizer.load(os.path.join(cache, f"spm_{data_cfg.tokenizer_model}_{data_cfg.vocab_size}.model"))
print("tokenizer vocab:", tok.vocab_size)

# make_dataloader(cfg, tok, SOURCE_path, TARGET_path, ...): the two path args are positional,
# source then target. for the reverse en->hi model source = English, target = Hindi --
# the SWAP vs 02_train (which passed hi as source, en as target).
train_loader = make_dataloader(data_cfg, tok,
    os.path.join(cache, "train.labse.en"), os.path.join(cache, "train.labse.hi"),
    max_tokens=train_cfg.max_tokens, train=True)
dev_loader = make_dataloader(data_cfg, tok,
    os.path.join(cache, "dev.clean.en"), os.path.join(cache, "dev.clean.hi"),
    max_tokens=train_cfg.max_tokens, train=False)
print("train batches:", len(train_loader), "| dev batches:", len(dev_loader))

tokenizer vocab: 16000
train batches: 2693 | dev batches: 3


## 4. Build the model

In [8]:
from nmt.model.transformer import build_model

model = build_model(model_cfg, tok)     # copies vocab_size + special ids from the tokenizer
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params/1e6:.1f}M | d_model={model_cfg.d_model} heads={model_cfg.n_heads} "
      f"layers={model_cfg.n_enc_layers}+{model_cfg.n_dec_layers} tied={model_cfg.tie_embeddings}")

params: 52.3M | d_model=512 heads=8 layers=6+6 tied=True


## 5. (Optional) Live TensorBoard

Run before training to watch `train/loss`, `train/lr`, `train/grad_norm`, `train/tok_per_s`, and `val/nll` update live.

In [9]:
# TB event files go to out_dir on Drive/backtranslate; live updates off a Drive mount can lag a bit
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/hindi-translator/checkpoints/backtranslate

<IPython.core.display.Javascript object>

## 6. Train

Auto-resumes if `step_*.pt` already exist in the Drive `out_dir`. Watch `train/loss` fall from ~log(16000)=9.7 (random) downward and `val/nll` (now Hindi prediction) track down. Saves `best.pt` + the last 3 rolling `step_*.pt` to Drive. A disconnect is fine — reconnect and re-run to resume.

In [10]:
from nmt.train.loop import Trainer

trainer = Trainer(train_cfg, model, train_loader, dev_loader, tok, out_dir=CKPT_ROOT)
trainer.train()
print("done. step:", trainer.step, "| best dev nll:", trainer.best)

done. step: 56000 | best dev nll: 2.1668359489881257


## 7. Next

This run's `best.pt` is now on Drive at `checkpoints/backtranslate`. That reverse en→hi model is what `03_backtranslation` will use to turn monolingual English into synthetic Hindi (tagged `<bt>`), which then gets merged into the forward training set for a second `02_train` run.